In [81]:
import seaborn as sns
import numpy as np
import pandas as pd
import re
from sklearn.model_selection import train_test_split

# Loading Data

In [82]:
dataset = pd.read_csv('emails.csv')

# Preprocessing 

In [83]:
def lower(s):
    return s.lower()

lower_vectorized = np.vectorize(lower)

dataset['text'] = lower_vectorized(dataset['text'])

In [84]:
def remove_special_characters(s):
    special_characters = '!@#^*().,/=-+_`'
    for i in special_characters:
        s = s.replace(i, ' ')
    return s

def remove_white_spaces(s):
    return re.sub(r'\s+', ' ', s)

remove_white_spaces = np.vectorize(remove_white_spaces)

remove_characters_vectorized = np.vectorize(remove_special_characters)

In [85]:
def extract_subject(s):
    arr = s.split(':')
    return arr[1].strip()

extract_subject = np.vectorize(extract_subject)

dataset['text'] = remove_characters_vectorized(dataset['text'])
dataset['text'] = remove_white_spaces(dataset['text'])
dataset['subject'] = extract_subject(dataset['text'])

# Train/Test Split

In [86]:
X = pd.DataFrame()
X['text'] = dataset['text']

X_train, X_test, y_train, y_test = train_test_split(X, dataset['spam'], test_size = 0.33)

x = pd.DataFrame()
x['text'] = X_train['text']
x['label'] = y_train

# Exploratory Data Analysis

In [88]:
dataset.describe()

,spam
count,5728.000000
mean,0.238827
std,0.426404
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


# BoW Matrix

In [150]:
all_words = set()
all_texts = X_train['text']

for text in all_texts:
    words = text.split(' ')
    for word in words:
        all_words.add(word)

all_words = list(all_words)

In [104]:
BoW = []

def word_count(s, word):
    series = []

    for string in s:
        series.append(string.count(word))
    return series

for word in all_words:
    BoW.append(word_count(all_texts, word))

In [108]:
BoW = np.array(BoW)

In [109]:
BoW.shape

(30251, 3837)

In [158]:
spam_texts = x[x['label'] == 1]['text']
non_spam_texts = x[x['label'] == 0]['text']

In [164]:
spam = []

def word_count(words, text):
    series = []
    for word in words:
        series.append(text.count(word))

    return series

for text in spam_texts:
    spam.append(word_count(all_words, text))

spam = np.array(spam)

non_spam = []

for text in non_spam_texts:
    non_spam.append(word_count(all_words, text))

non_spam = np.array(non_spam)

In [165]:
spam = spam.sum(axis=0)
non_spam = non_spam.sum(axis=0)

new_BoW = []
new_BoW.append(spam)
new_BoW.append(non_spam)

In [167]:
spam_chance = len(spam_texts)/(len(spam_texts)+len(non_spam_texts))
non_spam_chance = len(non_spam_texts)/(len(spam_texts)+len(non_spam_texts))

In [168]:
# First Row : y=1
# Second Row : y=0

conditional_probability = new_BoW.copy()
conditional_probability[0] = conditional_probability[0] / conditional_probability[0].sum()
conditional_probability[1] = conditional_probability[1] / conditional_probability[1].sum()

In [169]:
len(conditional_probability[0])

30251

In [170]:
def spam_probability(text):
    final = 1;
    
    for i in range(len(all_words)):
        word = all_words[i]
        if word in text:
            count = text.count(word)
            final *= (spam_chance * conditional_probability[0][i])

    return final

def non_spam_probability(text):
    final = 1;
    
    for i in range(len(all_words)):
        word = all_words[i]
        if word in text:
            count = text.count(word)
            final *= (non_spam_chance * conditional_probability[1][i])

    return final

def is_spam(text):
    return spam_probability(text) > non_spam_probability(text)

In [171]:
y_predicted = X_test.copy()

is_spam = np.vectorize(is_spam)

y_predicted = is_spam(y_predicted)

In [180]:
np.squeeze(y_predicted)

array([False, False, False, ..., False, False, False])

In [194]:
accuracy = []
y_test = np.array(y_test)

def bool_to_int(b):
    if b:
        return 1

    return 0

for i in range(len(y_predicted)):
    if bool_to_int(y_predicted[i]) == y_test[i]:
        accuracy.append(1)

    else:
        accuracy.append(0)

accuracy = np.array(accuracy)
accuracy = int(accuracy.sum()/len(accuracy) * 100)

In [195]:
accuracy

76